# 73장 · 벡터의 L2 거리와 pgvector 검색

18장에서 비교한 거리를 이번에는 SQL 검색 결과와 대조합니다.

질의 벡터 [1, 1, 0]과 문서 p2의 벡터 [1, 0, 1] 사이의 제곱거리는 2이고 거리는 약 1.414입니다. 파이썬 기준값을 먼저 구하고, pgvector 환경이 준비된 경우 같은 벡터로 SQL 조회 결과를 비교합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 어휘 벡터와 유클리드 거리

이 예제는 ‘과제’, ‘제출’, ‘검토’가 원문에 들어 있으면 해당 칸을 1로 둡니다. 단어 포함 여부를 보는 수업용 벡터라서 의미가 비슷해도 표현이 다른 단어는 잡지 못합니다.

**단어 벡터의 L2 거리**　과제 제출을 질의하면 p1까지의 거리는 0, p2까지는 약 1.414입니다.

[웹 교재에서 열기](https://langchain-lecture.vercel.app/chapter73/index.html#practice-space)


### 세 좌표에 의미를 부여하는 어휘 순서

<p>이 예제의 좌표는 과제, 제출, 검토의 순서입니다. 단어가 문장에 있으면 1.0, 없으면 0.0입니다. 문서와 질문에 같은 순서를 적용해야 대응하는 좌표끼리 거리를 잴 수 있습니다.</p>


### 셀 01 · 유클리드 거리 계산

검색 질문과 두 문서에 같은 규칙을 적용합니다. 질문과 거리가 가까운 문서부터 정렬합니다.

p2에서 다른 좌표가 두 개이고 각 차이의 크기가 1이면 제곱합은 1²+1²=2입니다. L2 거리는 여기에 제곱근을 취한 약 1.414입니다. 같은 벡터라도 거리와 거리의 제곱은 숫자가 다릅니다.


In [1]:
from math import dist
terms = ["과제", "제출", "검토"]
def embed(text):
    return [float(word in text) for word in terms]

query = embed("과제 제출")
texts = {"p1": "과제 제출 장소는 과제함입니다.", "p2": "과제 검토 기준을 읽으세요."}
ranked = sorted((dist(query, embed(text)), key) for key, text in texts.items())
print([(key, round(distance, 3)) for distance, key in ranked])

[('p1', 0.0), ('p2', 1.414)]


p1은 세 칸 모두 질문과 같아서 거리가 0입니다. p2는 제출과 검토 두 칸이 달라 L2 거리가 √2입니다. 거리를 제곱하면 2가 되지만 순서는 같습니다.


#### 같은 순위인데 거리 숫자가 달라지는 이유

<mark>pgvector의 거리 연산자와 math.dist는 제곱근까지 취한 L2 거리를 사용합니다.</mark> 질의와 p2는 두 좌표에서 각각 1만큼 다르므로 제곱거리 2에 해당하는 실제 거리는 약 1.414입니다. 18장의 검색 결과와 비교할 때에는 순위와 거리 단위를 따로 읽으세요.

선수 개념 · 18장 Chroma와 FAISS에서 검색 결과 대조하기

#### 같은 좌표를 제외하고 L2 거리 읽기

$$d(q,x)=\sqrt{\sum_i(q_i-x_i)^2}$$

두 문서와 질의의 첫 좌표인 과제 값은 모두 1입니다. 제출·검토 두 축만 보면 p1은 질의와 겹치고 p2는 가로와 세로로 한 칸씩 떨어져 있습니다. 대각선 길이는 √2이고, 소수 셋째 자리까지 쓰면 1.414입니다.



### L2 거리에서 1.414가 나오는 이유

<p>질문 [1, 1, 0]에서 p2의 [1, 0, 1]을 빼면 차이는 [0, 1, -1]입니다. 제곱의 합은 2이고 제곱근은 약 1.414입니다. p1은 질문과 세 좌표가 같아 거리 0으로 먼저 나옵니다.</p>


## 문서 ID와 거리로 pgvector 검색 결과 검산하기

Compose로 연결한 DB에서 대출 문의에 가까운 문서를 찾습니다. 먼저 70장의 실제 답과 반납 안내를 짧은 두 좌표로 표현해 같은 질문으로 거리를 계산해 보세요.

노트북을 같은 수업 폴더에 모아 열거나 압축을 푼 장별 폴더에서 실행하세요. 시작 셀은 service-lab/compose.json을 먼저 엽니다. 파일이 없을 때만 셀에 적힌 복귀 예시를 읽습니다.

In [ ]:
from pathlib import Path
import json
here = Path.cwd()
# 각 장 폴더에서 열거나 노트북을 한 폴더에 모아 두어도 같은 실습 폴더를 씁니다.
course = here.parent if here.name in ("daily",) or (here.name.startswith("chapter") and here.name[7:].isdigit()) else here
lab = course / "service-lab"
lab.mkdir(exist_ok=True)
incoming = course / "service-lab/compose.json"
if incoming.exists():
    previous = json.loads(incoming.read_text(encoding="utf-8"))
    print("내 실습 파일", incoming.name)
else:
    previous = {'route': '/turn',
     'turn': {'id': 'loan-demo',
              'question': '대출 기간은 며칠인가요?',
              'answer': '대출 기간은 14일입니다.',
              'source': 'loan-01'},
     'api_port': 8080,
     'db_service': 'db',
     'db_port': 55432}
    print("복귀 예시", incoming.name)

내 실습 파일 compose.json


## 문서 ID와 거리로 pgvector 검색 결과 검산하기 실습

앞에서 확인한 최소 예제에 내 실습 파일의 값을 넣습니다.

In [ ]:
from math import dist
terms = ["대출", "반납"]
def vector(text):
    return [int(term in text) for term in terms]
docs = {previous["turn"]["source"]: previous["turn"]["answer"], "return-note": "반납은 자료실에서 받습니다."}
question = previous["turn"]["question"]
ranked = sorted((dist(vector(question), vector(text)), key) for key, text in docs.items())
result = {"question": question, "top_id": ranked[0][1], "ranked": ranked,
          "turn": previous["turn"], "route": previous["route"], "api_port": previous["api_port"]}
print([(key, round(distance, 2)) for distance, key in ranked])

[('loan-01', 0.0), ('return-note', 1.41)]


좌표는 각각 대출과 반납이라는 단어의 존재 여부입니다. 계산한 거리가 0이면 두 좌표에서 질문과 문서의 값이 각각 같습니다. 이 로컬 계산을 아래 pgvector 조회의 정렬 순서와 대조합니다. 단어가 바뀌면 같은 뜻이어도 두 좌표가 달라질 수 있습니다.

### 같은 두 문서를 PostgreSQL로 조회하기

72장의 DB에 vector 확장을 준비하고 수업용 연결 문자열을 PGVECTOR_DSN에 설정하세요. 이 셀은 제공한 두 문서를 VALUES로 넘겨 실제 <-> 거리 정렬을 요청합니다. 출력 ID를 바로 위의 로컬 계산과 대조하세요.

In [ ]:
import os
if os.environ.get("PGVECTOR_DSN"):
    import psycopg
    sql = """SELECT id FROM (VALUES (%s, %s::vector), (%s, %s::vector)) AS d(id, v)
             ORDER BY v <-> %s::vector, id"""
    entries = list(docs.items())
    args = [entries[0][0], str(vector(entries[0][1])), entries[1][0], str(vector(entries[1][1])), str(vector(question))]
    with psycopg.connect(os.environ["PGVECTOR_DSN"]) as connection:
        ids = [row[0] for row in connection.execute(sql, args)]
    print(ids, ids == [key for distance, key in ranked])
else:
    print("DB 연결 전 · 위 셀의 로컬 거리만 계산했습니다.")

DB 연결 전 · 위 셀의 로컬 거리만 계산했습니다.


### 직접 해 보기

질문을 “반납은 어디에서 하나요?”로 바꿔 첫 문서 ID를 찾으세요. 원래 질문의 순위는 다음 장의 점검을 위해 그대로 보관하세요.

In [ ]:
query_check = "반납은 어디에서 하나요?"
# 바뀐 질문으로 거리와 ID를 정렬하세요.

<details class="notebook-answer"><summary>정답 코드와 실행 결과</summary><pre><code>query_check = &quot;반납은 어디에서 하나요?&quot;
print(min((dist(vector(query_check), vector(text)), key) for key, text in docs.items())[1])</code></pre><pre>return-note</pre><p>반납을 포함하는 두 번째 좌표만 같아져 반납 안내가 먼저 나옵니다.</p></details>

### 74장에서 다시 열 파일

방금 계산한 값을 service-lab/nearest.json에 저장합니다. 같은 장에서 조건을 바꾸고 다시 저장하면 다음 장도 그 값을 읽습니다.

In [ ]:
destination = lab / "nearest.json"
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(destination.name)

nearest.json


### 과제 02 · 질의 변경에 따른 거리 비교

질문을 ‘과제 검토’로 바꿔 두 문서의 순위와 거리를 출력하세요. 첫 검색에서 p2가 뒤에 있었던 이유를 새 결과와 대조하세요.


In [2]:
new_query = embed('과제 검토')
# 각 문서의 L2 거리를 계산하고 가까운 순서대로 키와 거리를 출력하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>new_query = embed(&quot;과제 검토&quot;)
distances = [(dist(new_query, embed(text)), key) for key, text in texts.items()]
for distance, key in sorted(distances):
    print(key, round(distance, 3))</code></pre>
<p>실행 결과</p><pre><code>p2 0.0
p1 1.414</code></pre>
<p>새 질의는 [1, 0, 1]이어서 p2와 일치합니다. 두 문서는 그대로 두고 질의만 바꿨으므로 순위가 뒤집힌 원인을 어휘 좌표에서 찾을 수 있습니다.</p>
</details>


## 선택 실습 · pgvector 거리순 정렬

PostgreSQL에서 vector(3) 열은 세 값을 받습니다. <-> 연산자는 L2 거리를 반환하므로 작은 순서대로 정렬합니다. 실제 서비스는 여기에 소유자·허용 문서 범위를 걸고 청크 ID로 원문 표를 연결합니다.


### 질문 벡터를 매개변수로 전달하기

<p>선택 셀은 %s::vector 자리에 질문 벡터를 별도 매개변수로 전달합니다. &lt;-&gt; 연산의 raw_l2를 오름차순으로 정렬하고, 거리가 같을 때는 문서 id를 두 번째 정렬 기준으로 사용합니다.</p>


### 셀 03 · 선택 실습 · PostgreSQL 벡터 조회

강사가 읽기 가능한 pgvector 수업 DB를 준비했으면 run_postgres를 켭니다. 연결 설정은 수업 커널의 표준 PG 환경변수를 사용합니다. 아래 SELECT는 제공한 두 벡터만 계산하며 테이블을 변경하지 않습니다.


In [3]:
run_postgres = False
sql = """WITH docs(id, embedding) AS (
  VALUES ('p1', '[1,1,0]'::vector(3)), ('p2', '[1,0,1]'::vector(3))
) SELECT id, embedding <-> %s::vector AS raw_l2
FROM docs ORDER BY raw_l2, id"""
if run_postgres:
    import psycopg
    with psycopg.connect(connect_timeout=3) as pg:
        rows = pg.execute(sql, ("[1,1,0]",)).fetchall()
        print([(key, round(distance, 3)) for key, distance in rows])
else:
    print("PostgreSQL 연결을 선택하지 않았습니다.")

PostgreSQL 연결을 선택하지 않았습니다.


run_postgres가 False이면 SQL 쿼리는 실행하지 않습니다. vector 확장을 설치한 DB에서 psycopg 연결을 준비한 경우에만 앞 셀의 p1·p2 거리와 실제 SQL 반환값을 비교할 수 있습니다.


### Python 계산과 PostgreSQL 실행

<p>run_postgres가 False인 기본 상태에서는 SQL 문자열만 준비합니다. PostgreSQL 연결과 pgvector 확장을 갖춘 환경에서만 선택 셀의 SQL을 실행해 Python의 거리 순서와 직접 비교할 수 있습니다.</p>


## 사용자 필터를 적용한 뒤 가장 가까운 문서 찾기

두 문서 집합에 좌표가 같은 자료가 있어도 사용자가 열 수 있는 범위 안에서만 검색해야 합니다. 3차원 어휘 벡터를 그대로 두고 필터와 상위 1개 선택의 순서를 바꿔 봅니다.

### 셀 05 · 전체에서 한 건을 먼저 뽑으면 사라지는 문서

다른 강좌의 문서가 질문과 정확히 일치합니다. 전체 1등을 꺼낸 뒤 내 강좌 것만 남기면 출력에 무엇이 남을지 예상하세요.

In [4]:
from math import dist
query_vector = [1, 1, 0]
scoped_docs = [
    {"id": "A:p1", "course": "A", "v": [1, 1, 0]},
    {"id": "B:p3", "course": "B", "v": [1, 0, 0]},
]
nearest = min(scoped_docs, key=lambda row: dist(query_vector, row["v"]))
print(nearest["id"], [row["id"] for row in [nearest] if row["course"] == "B"])

A:p1 []


A:p1 []가 나옵니다. 검색 뒤에 A를 숨겼지만 B:p3까지 후보에서 사라졌습니다. 1건의 한도를 적용하기 전으로 돌아가야 B에서 가장 가까운 한 건을 고를 수 있습니다.

### 과제 06 · 강좌 B 안에서 상위 한 건 다시 계산하기

강좌와 k를 받는 함수를 작성하세요. 먼저 범위를 좁힌 뒤 L2 거리와 문서 ID 순으로 정렬해 B의 첫 문서와 거리를 출력하세요. 이 결과를 PostgreSQL 쿼리의 기대값으로 사용합니다.

In [5]:
# scoped_docs를 course로 걸러 낸 뒤 거리와 id를 계산하세요.
# B에서 상위 한 건을 뽑아 (거리, 문서 ID) 순서로 출력하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>def nearest_in_course(docs, course, query, k):
    visible = [row for row in docs if row[&quot;course&quot;] == course]
    scores = [(dist(query, row[&quot;v&quot;]), row[&quot;id&quot;]) for row in visible]
    return sorted(scores)[:k]

print(nearest_in_course(scoped_docs, &quot;B&quot;, query_vector, 1))</code></pre>
<p>실행 결과</p><pre><code>[(1.0, &#x27;B:p3&#x27;)]</code></pre>
<p>B:p3가 거리 1.0으로 돌아옵니다. 질문과 다른 좌표는 두 번째 값 한 칸입니다. 범위를 먼저 적용한 뒤 비교한 값이므로 전체에서 가장 가까운 문서 A:p1과 달라도 검색 조건에 맞습니다.</p>
</details>

### 셀 07 · 선택 실습 · 같은 입력을 PostgreSQL에서 비교하기

셀 03을 실행해 run_postgres 설정을 준비하세요. 앞 절에서 run_postgres를 켜고 pgvector 사용이 가능한 수업 DB에 연결했다면 같은 두 행을 쿼리에 직접 넣어 보세요. 임시 VALUES를 사용하므로 기존 테이블을 수정하지 않습니다.

In [6]:
scoped_sql = """
WITH docs(id, course, embedding) AS (
    VALUES ('A:p1', 'A', '[1,1,0]'::vector(3)),
           ('B:p3', 'B', '[1,0,0]'::vector(3))
)
SELECT embedding <-> %s::vector AS distance, id
FROM docs WHERE course = %s ORDER BY distance, id LIMIT %s
"""
if run_postgres:
    import psycopg
    with psycopg.connect(connect_timeout=3) as pg:
        print(pg.execute(scoped_sql, ("[1,1,0]", "B", 1)).fetchall())
else:
    print("pgvector 비교는 수업 DB 연결 후 실행하세요.")

pgvector 비교는 수업 DB 연결 후 실행하세요.


DB를 연결했을 때 확인할 값은 1.0과 B:p3입니다. WHERE가 강좌를 제한하고 ORDER BY와 LIMIT이 그 안에서 문서를 고릅니다. 기본 출력은 연결 안내이므로 SQL 실행 결과로 기록하지 않습니다.

<mark>공개 범위를 정한 뒤 그 안에서 top-k를 선택하세요.</mark> 전체 문서 한 건을 골라 필터링하면 빈 목록이 나왔지만, 같은 입력이라도 범위를 먼저 좁히면 B:p3를 찾았습니다.

실제 Retriever의 회귀 질문에 강좌 ID와 기대 문서 ID를 함께 남겨, 저장소를 바꾼 뒤에도 같은 조건을 검사하세요.

## 돌아보기

질문을 ‘마감일’로 바꾸면 세 어휘가 모두 빠져 [0, 0, 0]이 됩니다. 두 문서까지의 거리가 같을 때 코드가 먼저 돌려주는 ID는 검색어와 뜻이 더 비슷하다는 근거가 되지 않습니다.
